# Synthetic Market Dynamics — Exploration Notebook

This notebook walks through each component of the project interactively.
Run the cells in order, or jump to whatever part you're interested in.

For a full pipeline run, use `src/main.py` instead.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (14, 5)

print('Ready')

## 1. Generate synthetic price series

In [ ]:
from time_series_model import generate_price_series

df = generate_price_series(n=1000, seed=42)
print(df.head())
print(f'\nShape: {df.shape}')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))
ax1.plot(df['date'], df['price'], lw=0.8)
ax1.set_title('Synthetic Price')
ax2.bar(df['date'], df['log_ret'] * 100, color=['red' if r < 0 else 'green' for r in df['log_ret']], width=0.5, alpha=0.7)
ax2.set_title('Log Returns (%)')
plt.tight_layout()
plt.show()

## 2. Fit EGARCH(1,1)

In [ ]:
from time_series_model import EGARCH

egarch = EGARCH()
egarch.fit(df['log_ret'].values)

print('EGARCH parameters:')
for k, v in egarch.params.items():
    print(f'  {k:8s} = {v:.4f}')
print(f'  Converged: {egarch.converged}')

In [ ]:
# Plot fitted vol vs true DGP vol
plt.figure(figsize=(14, 4))
plt.plot(df['date'], egarch.fitted_vol * 100, label='EGARCH fitted', lw=0.8)
plt.plot(df['date'], df['true_vol'] * 100, label='True DGP vol', lw=0.8, linestyle='--', alpha=0.7)
plt.title('Conditional Volatility: Fitted vs True')
plt.ylabel('Daily vol (%)')
plt.legend()
plt.show()

## 3. VaR and CVaR

In [ ]:
from time_series_model import compute_var_cvar

var, cvar = compute_var_cvar(df['log_ret'].values)
print(f'VaR  (95%): {var*100:.2f}%')
print(f'CVaR (95%): {cvar*100:.2f}%')

# Quick histogram
rets = df['log_ret'].values * 100
plt.figure(figsize=(10, 4))
plt.hist(rets, bins=70, alpha=0.6, color='steelblue', density=True, edgecolor='none')
plt.axvline(var*100, color='red', lw=1.5, linestyle='--', label=f'VaR = {var*100:.2f}%')
plt.axvline(cvar*100, color='orange', lw=1.5, linestyle=':', label=f'CVaR = {cvar*100:.2f}%')
plt.title('Return Distribution with Risk Thresholds')
plt.xlabel('Daily return (%)')
plt.legend()
plt.show()

## 4. 30-Day Price Forecast

In [ ]:
from time_series_model import ARIMA, forecast_prices

arima = ARIMA(p=0, d=0, q=0)
arima.fit(df['log_ret'])

fc = forecast_prices(df, arima, egarch, steps=30, n_simulations=300)

print(f"Mean forecast sigma: {fc['mean_vol']:.4f}")
print(f"Median exit price: ${fc['median'][-1]:.2f}")

In [ ]:
last_date = df['date'].iloc[-1]
fcast_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=30, freq='B')

plt.figure(figsize=(14, 5))
plt.plot(df['date'].tail(60), df['price'].tail(60), color='steelblue', lw=1.2, label='Historical')
for path in fc['paths'][:60]:
    plt.plot(fcast_dates, path, color='steelblue', alpha=0.05, lw=0.5)
plt.fill_between(fcast_dates, fc['lower5'], fc['upper95'], alpha=0.15, color='steelblue', label='90% band')
plt.plot(fcast_dates, fc['median'], color='lime', lw=1.5, label='Median')
plt.axvline(last_date, color='yellow', lw=1, linestyle='--')
plt.title('30-Day Price Forecast')
plt.legend()
plt.show()

## 5. HFT Simulation (run with fewer trades for speed in notebook)

In [ ]:
from hft_market_simulator import run_simulation, bin_to_30min, cluster_bins

# Use 500k trades for interactive exploration (full 5M in main.py)
trades = run_simulation(n_trades=500_000, seed=42)
print(trades.head())
print(f'\nTotal P&L: {trades["pnl"].sum():,.0f}')

In [ ]:
binned = bin_to_30min(trades)
print(f'Bins: {len(binned)}')

plt.figure(figsize=(14, 4))
plt.fill_between(binned.index, 0, binned['volume'] / 1e6, alpha=0.5)
plt.title('30-Min Volume Bins')
plt.ylabel('Volume (M)')
plt.show()

In [ ]:
binned_cl, cluster_summary, _, _ = cluster_bins(binned, n_clusters=6)

print('Cluster summary:')
print(cluster_summary[['rank_label', 'n_bins', 'avg_trades', 'total_volume', 'avg_ofi']].to_string(index=False))

## 6. Full visualization suite

Run `src/main.py` to generate all 6 publication-ready figures saved to `outputs/`.